# Causal Hybrid ML Trading Strategy

Implements a Cross-asset Hybrid Machione Learning forecasting model for the 5-day probable return of the S&P 500. This model helps forecast the regime (Normal trending or Crash) for the next week and 
forecast significant Drawdowns in S&P500 which can be translated to a trading stratgy.

In [4]:
# Import all relevant modules

import os
import pandas as pd
import numpy as np
import yfinance as yf
from datetime import datetime
import logging
from typing import Dict, List, Optional, Tuple
import pickle
from scipy.stats import entropy, skew, kurtosis
import warnings

# ML Libraries
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (
    roc_auc_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

try:
    from xgboost import XGBClassifier
except ImportError:
    XGBClassifier = None
    
try:
    from catboost import CatBoostClassifier
except ImportError:
    CatBoostClassifier = None

import joblib
    
warnings.filterwarnings('ignore')
logger = logging.getLogger(__name__)



 1- Import the data 

In [5]:
class YahooFinanceFetcher:
    """
    Fetches historical price data from Yahoo Finance.
    
    Supports:
    - Batch data retrieval
    - Caching to avoid redundant API calls
    - Data validation
    - Missing value handling
    """
    
    def __init__(self, config: dict, cache_dir: str = "./data/raw"):
        """
        Initialize data fetcher.
        
        Args:
            config: Configuration dictionary with data settings
            cache_dir: Directory for caching downloaded data
        """
        self.config = config['data']
        self.cache_dir = cache_dir
        self.start_date = self.config.get('start_date', '2005-01-01')
        self.end_date = self.config.get('end_date', datetime.now().strftime('%Y-%m-%d'))
        self.symbols = self.config.get('symbols', [])
        self.cache_enabled = self.config.get('cache_data', True)
        
        os.makedirs(cache_dir, exist_ok=True)
        logger.info(f"Data fetcher initialized - Period: {self.start_date} to {self.end_date}")
    
    def _get_cache_path(self, symbol: str) -> str:
        """Get cache file path for symbol."""
        return os.path.join(self.cache_dir, f"{symbol}_{self.start_date}_{self.end_date}.pkl")
    
    def _load_from_cache(self, symbol: str) -> Optional[pd.DataFrame]:
        """Load data from cache if exists."""
        if not self.cache_enabled:
            return None
        
        cache_path = self._get_cache_path(symbol)
        if os.path.exists(cache_path):
            try:
                with open(cache_path, 'rb') as f:
                    data = pickle.load(f)
                logger.debug(f"Loaded {symbol} from cache")
                return data
            except Exception as e:
                logger.warning(f"Cache read failed for {symbol}: {e}")
                return None
        return None
    
    def _save_to_cache(self, symbol: str, data: pd.DataFrame) -> None:
        """Save data to cache."""
        if not self.cache_enabled:
            return
        
        try:
            cache_path = self._get_cache_path(symbol)
            with open(cache_path, 'wb') as f:
                pickle.dump(data, f)
            logger.debug(f"Cached {symbol}")
        except Exception as e:
            logger.warning(f"Cache write failed for {symbol}: {e}")
    
    def fetch_single(self, symbol: str, progress: bool = True) -> pd.DataFrame:
        """
        Fetch historical data for single symbol.
        
        Args:
            symbol: Ticker symbol
            progress: Show download progress
            
        Returns:
            DataFrame with OHLCV data
        """
        # Try cache first
        cached_data = self._load_from_cache(symbol)
        if cached_data is not None:
            return cached_data
        
        logger.info(f"Downloading {symbol}...")
        try:
            data = yf.download(
                symbol,
                start=self.start_date,
                end=self.end_date,
                progress=progress
            )
            
            if data.empty:
                logger.warning(f"No data returned for {symbol}")
                return pd.DataFrame()
            
            data['Symbol'] = symbol
            self._save_to_cache(symbol, data)
            logger.info(f"✓ Downloaded {symbol}: {len(data)} records")
            return data
            
        except Exception as e:
            logger.error(f"Failed to download {symbol}: {e}")
            return pd.DataFrame()
    
    def fetch_historical_data(self) -> Dict[str, pd.DataFrame]:
        """
        Fetch historical data for all symbols in universe.
        
        Returns:
            Dictionary mapping symbol -> DataFrame
        """
        data = {}
        failed_symbols = []
        
        for symbol in self.symbols:
            df = self.fetch_single(symbol, progress=False)
            if df.empty:
                failed_symbols.append(symbol)
            else:
                data[symbol] = df
        
        if failed_symbols:
            logger.warning(f"Failed to fetch: {failed_symbols}")
        
        logger.info(f"✓ Successfully fetched {len(data)}/{len(self.symbols)} symbols")
        return data
    
    def fetch_combined(self) -> pd.DataFrame:
        """
        Fetch and combine data for all symbols into single DataFrame.
        
        Returns:
            DataFrame with 'Close' prices for all symbols as columns
        """
        data = self.fetch_historical_data()
        
        combined = pd.DataFrame()
        for symbol, df in data.items():
            # yfinance 1.0 returns 'Close' column (not 'Adj Close')
            if 'Close' in df.columns:
                combined[symbol] = df['Close']
            elif 'Adj Close' in df.columns:  # Fallback for older versions
                combined[symbol] = df['Adj Close']
        
        combined = combined.sort_index()
        logger.info(f"Combined data shape: {combined.shape}")
        if len(combined) > 0:
            logger.info(f"Date range: {combined.index[0]} to {combined.index[-1]}")
        else:
            logger.warning("Combined DataFrame is empty - no data to process")
        
        return combined
    
    def save_raw_data(self, data: Dict[str, pd.DataFrame], output_dir: str = "./data/raw") -> None:
        """Save raw data to CSV files."""
        os.makedirs(output_dir, exist_ok=True)
        
        for symbol, df in data.items():
            path = os.path.join(output_dir, f"{symbol}.csv")
            df.to_csv(path)
            logger.info(f"Saved {symbol} to {path}")


class CSVDataLoader:
    """Load data from local CSV files (alternative to Yahoo Finance)."""
    
    def __init__(self, data_dir: str = "./data/raw"):
        """Initialize CSV loader."""
        self.data_dir = data_dir
    
    def load_symbol(self, symbol: str) -> pd.DataFrame:
        """Load CSV for single symbol."""
        path = os.path.join(self.data_dir, f"{symbol}.csv")
        if not os.path.exists(path):
            raise FileNotFoundError(f"File not found: {path}")
        
        df = pd.read_csv(path, index_col=0, parse_dates=True)
        logger.info(f"Loaded {symbol} from {path}")
        return df
    
    def load_all(self, symbols: List[str]) -> Dict[str, pd.DataFrame]:
        """Load all symbols."""
        data = {}
        for symbol in symbols:
            try:
                data[symbol] = self.load_symbol(symbol)
            except Exception as e:
                logger.warning(f"Failed to load {symbol}: {e}")
        return data


# 2- Feature Engineering

Opposed to technical indicators, this ensamble focuses on statistical measures for trend and dispersion (volatility) and Informational Theory measures of several asset classes besides the very S&P500. . 

In [6]:
"""Implements all 178+ features as described in the paper:
- Time-series moments (volatility, skewness, kurtosis, entropy)
- Hurst exponent (multi-scale persistence)
- Cross-asset relationships (beta, correlation)
- Information-theoretic measures (KL divergence)
"""


warnings.filterwarnings('ignore')
logger = logging.getLogger(__name__)


class FeatureEngineer:
    """
    Comprehensive feature engineering for systematic alpha model.
    
    Implements all feature categories from the research paper.
    """
    
    def __init__(self, config: dict):
        """Initialize feature engineer with configuration."""
        self.config = config.get('features', {})
        self.rolling_windows = self.config.get('rolling_windows', [21, 63])
        self.hurst_scales = self.config.get('hurst_scales', [16, 64, 256])
        self.entropy_bins = self.config.get('entropy_bins', 30)
        self.target_drawdown = self.config.get('target_drawdown', 0.01)
        self.target_horizon = self.config.get('target_horizon', 5)
        self.target_symbol = 'SPY'
    
    def compute_returns(self, prices: pd.DataFrame) -> pd.DataFrame:
        """Compute log returns from price series."""
        returns = np.log(prices / prices.shift(1))
        return returns.dropna()
    
    def compute_volatility(self, returns: pd.Series, window: int) -> pd.Series:
        """Rolling standard deviation."""
        return returns.rolling(window=window).std()
    
    def compute_skewness(self, returns: pd.Series, window: int) -> pd.Series:
        """Rolling skewness (3rd moment)."""
        return returns.rolling(window=window).apply(
            lambda x: skew(x, bias=True) if len(x) > 2 else np.nan
        )
    
    def compute_kurtosis(self, returns: pd.Series, window: int) -> pd.Series:
        """Rolling excess kurtosis (4th moment - 3)."""
        return returns.rolling(window=window).apply(
            lambda x: kurtosis(x, bias=True, fisher=True) if len(x) > 3 else np.nan
        )
    
    def compute_entropy(self, returns: pd.Series, window: int, bins: int = 30) -> pd.Series:
        """Rolling Shannon entropy of return distribution."""
        def _entropy(x):
            if len(x) < bins:
                return np.nan
            hist, _ = np.histogram(x, bins=bins)
            hist = hist / hist.sum()
            hist = hist[hist > 0]
            return -np.sum(hist * np.log(hist))
        
        return returns.rolling(window=window).apply(_entropy)
    
    def compute_hurst_exponent(self, returns: pd.Series, scale: int) -> pd.Series:
        """
        Compute Hurst exponent using rescaled range analysis.
        
        Higher Hurst (>0.5): trending/persistent
        Lower Hurst (<0.5): mean-reverting
        H=0.5: random walk
        
        Args:
            returns: Return series
            scale: Time scale for analysis (16, 64, 256)
            
        Returns:
            Series of Hurst exponents
        """
        def _hurst(x):
            if len(x) < scale * 2:
                return np.nan
            
            # Cumulative sum
            cumsum = np.cumsum(x - x.mean())
            
            # Range
            max_val = np.maximum.accumulate(cumsum)
            min_val = np.minimum.accumulate(cumsum)
            range_val = max_val - min_val
            
            # Standard deviation
            std_val = np.std(x, ddof=1)
            if std_val == 0:
                return np.nan
            
            # Rescaled range
            r_s = range_val / std_val
            
            # Hurst exponent (H = log(R/S) / log(N))
            n = np.arange(1, len(r_s) + 1)
            h = np.log(r_s) / np.log(n)
            return np.mean(h[-min(scale, len(h)):])
        
        return returns.rolling(window=scale * 2).apply(_hurst)
    
    def compute_rolling_beta(self, asset_returns: pd.Series,
                            market_returns: pd.Series, window: int) -> pd.Series:
        """Compute rolling beta vs market (SPY).
        
        Beta = Cov(asset, market) / Var(market)
        Uses pandas optimized built-in methods for best performance.
        """
        # Beta = Covariance / Variance
        rolling_cov = asset_returns.rolling(window).cov(market_returns)
        rolling_var = market_returns.rolling(window).var()
        return rolling_cov / rolling_var
    
    def compute_rolling_correlation(self, asset_returns: pd.Series,
                                   market_returns: pd.Series, window: int) -> pd.Series:
        """Compute rolling correlation vs market."""
        return asset_returns.rolling(window=window).corr(market_returns)
    
    def compute_kl_divergence(self, returns: pd.Series, window_current: int,
                             window_reference: int, bins: int = 30) -> pd.Series:
        """
        Kullback-Leibler divergence vs reference distribution.
        
        Measures shift in return distribution (higher = more extreme).
        """
        def _kl_div(x):
            if len(x) < window_current:
                return np.nan
            
            current_returns = x[:window_current]
            ref_returns = x[window_current:]
            
            if len(ref_returns) < bins:
                return np.nan
            
            # Histograms
            hist_curr, edges = np.histogram(current_returns, bins=bins)
            hist_ref, _ = np.histogram(ref_returns, bins=edges)
            
            # Normalize
            p = hist_curr / hist_curr.sum()
            q = hist_ref / hist_ref.sum()
            
            # KL divergence with smoothing
            p = p + 1e-10
            q = q + 1e-10
            
            return np.sum(p * np.log(p / q))
        
        combined = pd.concat([returns] * 2, axis=1).T
        return returns.rolling(window=window_current + window_reference).apply(_kl_div)
    
    def create_target_variable(self, prices: pd.DataFrame) -> pd.Series:
        """
        Create binary target: 1 if 5-day SPY drawdown >= 1%, else 0.
        
        y_t = 1 if sum(r_SPY[t+1:t+5]) <= -0.01, else 0
        """
        spy_returns = self.compute_returns(prices[[self.target_symbol]])
        
        # Rolling sum of returns over horizon
        cumsum_returns = spy_returns[self.target_symbol].rolling(
            window=self.target_horizon
        ).sum()
        
        # Shift forward to get future drawdown
        target = (cumsum_returns.shift(-self.target_horizon) <= -self.target_drawdown).astype(int)
        
        logger.info(f"Target variable created - Positive class: {target.sum()} samples")
        return target
    
    def engineer_all_features(self, prices: pd.DataFrame) -> pd.DataFrame:
        """
        Engineer all 178+ features from raw price data.
        
        Args:
            prices: DataFrame with adjusted close prices for all symbols
            
        Returns:
            DataFrame with engineered features (rows: dates, cols: features)
        """
        logger.info("Starting feature engineering...")
        
        returns = self.compute_returns(prices)
        features = pd.DataFrame(index=returns.index)
        
        feature_count = 0
        
        # 1. TIME-SERIES MOMENTS (Statistical Features)
        logger.info("Computing time-series moments...")
        for symbol in prices.columns:
            if symbol not in returns.columns:
                continue
            
            r = returns[symbol]
            
            for window in self.rolling_windows:
                # Volatility
                vol_feat = self.compute_volatility(r, window)
                features[f'{symbol}_vol_{window}d'] = vol_feat
                feature_count += 1
                
                # Skewness
                skew_feat = self.compute_skewness(r, window)
                features[f'{symbol}_skew_{window}d'] = skew_feat
                feature_count += 1
                
                # Kurtosis
                kurt_feat = self.compute_kurtosis(r, window)
                features[f'{symbol}_kurtosis_{window}d'] = kurt_feat
                feature_count += 1
                
                # Entropy
                ent_feat = self.compute_entropy(r, window, self.entropy_bins)
                features[f'{symbol}_entropy_{window}d'] = ent_feat
                feature_count += 1
        
        # 2. HURST EXPONENT (Persistence / Mean-Reversion Indicator)
        logger.info("Computing Hurst exponents...")
        for symbol in prices.columns:
            if symbol not in returns.columns:
                continue
            
            r = returns[symbol]
            
            for scale in self.hurst_scales:
                hurst_feat = self.compute_hurst_exponent(r, scale)
                features[f'{symbol}_hurst_{scale}'] = hurst_feat
                feature_count += 1
        
        # 3. CROSS-ASSET RELATIONSHIPS (vs SPY)
        logger.info("Computing cross-asset relationships...")
        spy_returns = returns[self.target_symbol]
        
        for symbol in prices.columns:
            if symbol not in returns.columns or symbol == self.target_symbol:
                continue
            
            asset_returns = returns[symbol]
            
            for window in self.rolling_windows:
                # Beta
                beta_feat = self.compute_rolling_beta(asset_returns, spy_returns, window)
                features[f'{symbol}_beta_SPY_{window}d'] = beta_feat
                feature_count += 1
                
                # Correlation
                corr_feat = self.compute_rolling_correlation(asset_returns, spy_returns, window)
                features[f'{symbol}_corr_SPY_{window}d'] = corr_feat
                feature_count += 1
        
        # 4. INFORMATION-THEORETIC MEASURES (KL Divergence)
        logger.info("Computing information-theoretic measures...")
        kl_window_pairs = [(21, 126), (63, 126)]  # current, reference window
        
        for symbol in prices.columns:
            if symbol not in returns.columns:
                continue
            
            r = returns[symbol]
            
            for w_curr, w_ref in kl_window_pairs:
                kl_feat = self.compute_kl_divergence(r, w_curr, w_ref, self.entropy_bins)
                features[f'{symbol}_kl_{w_curr}vs{w_ref}'] = kl_feat
                feature_count += 1
        
        logger.info(f"✓ Engineered {feature_count} features")
        logger.info(f"Feature matrix shape: {features.shape}")
        logger.info(f"Missing values before cleaning: {features.isna().sum().sum()}")
        
        # Remove columns with too many NaN values (>10%)
        nan_threshold = 0.1
        nan_pct = features.isna().mean()
        bad_features = nan_pct[nan_pct > nan_threshold].index
        if len(bad_features) > 0:
            logger.warning(f"Dropping {len(bad_features)} features with >{nan_threshold*100}% NaN values")
            features = features.drop(columns=bad_features)
        
        # Fill remaining NaN values: forward fill -> backward fill -> zero
        features = features.ffill().bfill().fillna(0)
        logger.info(f"Missing values after cleaning: {features.isna().sum().sum()}")
        logger.info(f"Final feature matrix shape: {features.shape}")
        
        return features
    
    def select_features_by_variance(self, features: pd.DataFrame,
                                   threshold: float = 1e-4) -> pd.DataFrame:
        """Remove low-variance features."""
        variances = features.var()
        low_var_features = variances[variances < threshold].index
        
        logger.info(f"Removing {len(low_var_features)} low-variance features")
        features = features.drop(columns=low_var_features)
        
        return features
    
    def select_features_by_correlation(self, features: pd.DataFrame,
                                      threshold: float = 0.95) -> pd.DataFrame:
        """Remove highly correlated feature pairs."""
        corr_matrix = features.corr().abs()
        
        upper = corr_matrix.where(
            np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
        )
        
        to_drop = [col for col in upper.columns if any(upper[col] > threshold)]
        
        logger.info(f"Removing {len(to_drop)} highly correlated features")
        features = features.drop(columns=to_drop)
        
        return features
    
    def select_top_features_by_mi(self, features: pd.DataFrame, target: pd.Series,
                                 n_features: int = 80) -> Tuple[pd.DataFrame, Dict]:
        """
        Select top N features by mutual information with target.
        """
        from sklearn.feature_selection import mutual_info_classif
        
        logger.info("Computing mutual information scores...")
        
        # Align indices
        valid_idx = features.index.intersection(target.index)
        X = features.loc[valid_idx]
        y = target.loc[valid_idx]
        
        # Compute MI
        mi_scores = mutual_info_classif(X, y, random_state=42)
        
        # Rank features
        mi_ranking = pd.Series(mi_scores, index=X.columns).sort_values(ascending=False)
        
        selected_features = mi_ranking.head(n_features)
        logger.info(f"Top features by MI:\n{selected_features.head(13)}")
        
        # Select
        X_selected = X[selected_features.index]
        
        logger.info(f"Selected {n_features} features based on mutual information")
        
        return X_selected, mi_ranking.to_dict()
    
    def standardize_features(self, features: pd.DataFrame, fit: bool = True) -> pd.DataFrame:
        """Standardize features to mean=0, std=1."""
        if fit:
            self.feature_mean = features.mean()
            self.feature_std = features.std()
        
        features_std = (features - self.feature_mean) / (self.feature_std + 1e-10)
        logger.info("Features standardized")
        
        return features_std


# 3- Model Set up
Hyperparameter optimization is conducted via a ROC maximization for binary classifiers of a Multi-layer perceptron ( non linearities and  hierarchies) and both an XG Boost and Cat Boost 

In [7]:

class HybridEnsemble:
    """
    Hybrid ensemble combining neural networks and tree-based models.
    
    Architecture:
    - MLP: Captures temporal non-linearities
    - XGBoost: Robust tree-based learning
    - CatBoost: Categorical handling and interpretability
    - Soft Voting: Averaged probability predictions
    """
    
    def __init__(self, config: dict):
        """Initialize ensemble."""
        self.config = config.get('model', {})
        self.ensemble_type = self.config.get('ensemble_type', 'soft_voting')
        self.random_state = self.config.get('random_state', 42)
        
        # Initialize base learners
        self.mlp = None
        self.xgboost = None
        self.catboost = None
        self.scaler = StandardScaler()
        
        self.trained_models = []
        self.feature_names = None
    
    def build_mlp(self) -> MLPClassifier:
        """Build MLP neural network."""
        hidden_layers = self.config.get('mlp_hidden_layers', [128, 64])
        dropout = self.config.get('mlp_dropout', 0.3)
        
        mlp = MLPClassifier(
            hidden_layer_sizes=tuple(hidden_layers),
            activation='relu',
            solver='adam',
            learning_rate='adaptive',
            learning_rate_init=0.001,
            max_iter=500,
            batch_size=32,
            early_stopping=True,
            validation_fraction=0.1,
            n_iter_no_change=10,
            random_state=self.random_state,
            verbose=0
        )
        
        logger.info(f"MLP built: {hidden_layers}")
        return mlp
    
    def build_xgboost(self) -> XGBClassifier:
        """Build XGBoost classifier."""
        if XGBClassifier is None:
            logger.warning("XGBoost not installed, skipping")
            return None
        
        xgb_params = self.config.get('xgboost_params', {})
        
        xgb = XGBClassifier(
            n_estimators=xgb_params.get('n_estimators', 200),
            max_depth=xgb_params.get('max_depth', 6),
            learning_rate=xgb_params.get('learning_rate', 0.1),
            subsample=xgb_params.get('subsample', 0.8),
            colsample_bytree=xgb_params.get('colsample_bytree', 0.8),
            objective='binary:logistic',
            eval_metric='logloss',
            random_state=self.random_state,
            verbosity=0,
            n_jobs=-1
        )
        
        logger.info("XGBoost built")
        return xgb
    
    def build_catboost(self) -> CatBoostClassifier:
        """Build CatBoost classifier."""
        if CatBoostClassifier is None:
            logger.warning("CatBoost not installed, skipping")
            return None
        
        cat_params = self.config.get('catboost_params', {})
        
        cat = CatBoostClassifier(
            iterations=cat_params.get('iterations', 200),
            depth=cat_params.get('depth', 6),
            learning_rate=cat_params.get('learning_rate', 0.1),
            loss_function='Logloss',
            verbose=False,
            random_state=self.random_state,
            thread_count=-1
        )
        
        logger.info("CatBoost built")
        return cat
    
    def fit(self, X: pd.DataFrame, y: pd.Series, validation_split: float = 0.2) -> None:
        """
        Train ensemble on data with time-series cross-validation.
        
        Args:
            X: Feature matrix (n_samples, n_features)
            y: Target vector (n_samples,)
            validation_split: Portion for validation
        """
        logger.info(f"Training ensemble on {X.shape[0]} samples, {X.shape[1]} features")
        
        self.feature_names = X.columns
        
        # Time series split
        n_train = int(len(X) * (1 - validation_split))
        X_train, X_val = X.iloc[:n_train], X.iloc[n_train:]
        y_train, y_val = y.iloc[:n_train], y.iloc[n_train:]
        
        # Standardize
        X_train_scaled = self.scaler.fit_transform(X_train)
        X_val_scaled = self.scaler.transform(X_val)
        
        logger.info(f"Train: {len(X_train)}, Val: {len(X_val)}")
        logger.info(f"Positive class: {y_train.sum()} ({y_train.sum()/len(y_train)*100:.1f}%)")
        
        # Train MLP
        logger.info("Training MLP...")
        self.mlp = self.build_mlp()
        self.mlp.fit(X_train_scaled, y_train)
        mlp_auc = roc_auc_score(y_val, self.mlp.predict_proba(X_val_scaled)[:, 1])
        logger.info(f"  MLP ROC-AUC (val): {mlp_auc:.4f}")
        self.trained_models.append(('MLP', self.mlp))
        
        # Train XGBoost
        logger.info("Training XGBoost...")
        self.xgboost = self.build_xgboost()
        if self.xgboost is not None:
            self.xgboost.fit(
                X_train, y_train,
                eval_set=[(X_val, y_val)],
                verbose=False
            )
            xgb_auc = roc_auc_score(y_val, self.xgboost.predict_proba(X_val)[:, 1])
            logger.info(f"  XGBoost ROC-AUC (val): {xgb_auc:.4f}")
            self.trained_models.append(('XGBoost', self.xgboost))
        
        # Train CatBoost
        logger.info("Training CatBoost...")
        self.catboost = self.build_catboost()
        if self.catboost is not None:
            self.catboost.fit(
                X_train, y_train,
                eval_set=(X_val, y_val),
                verbose=False
            )
            cat_auc = roc_auc_score(y_val, self.catboost.predict_proba(X_val)[:, 1])
            logger.info(f"  CatBoost ROC-AUC (val): {cat_auc:.4f}")
            self.trained_models.append(('CatBoost', self.catboost))
        
        logger.info("✓ Ensemble training complete")
    
    def predict_proba(self, X: pd.DataFrame) -> np.ndarray:
        """
        Predict crash probability via soft voting.
        
        Returns: Array of probabilities [n_samples, 2]
        """
        X_scaled = self.scaler.transform(X)
        
        probas = []
        
        # MLP prediction
        if self.mlp is not None:
            mlp_proba = self.mlp.predict_proba(X_scaled)[:, 1]
            probas.append(mlp_proba)
        
        # XGBoost prediction
        if self.xgboost is not None:
            xgb_proba = self.xgboost.predict_proba(X)[:, 1]
            probas.append(xgb_proba)
        
        # CatBoost prediction
        if self.catboost is not None:
            cat_proba = self.catboost.predict_proba(X)[:, 1]
            probas.append(cat_proba)
        
        # Soft voting: average probabilities
        ensemble_proba = np.mean(probas, axis=0)
        
        return np.column_stack([1 - ensemble_proba, ensemble_proba])
    
    def predict(self, X: pd.DataFrame, threshold: float = 0.5) -> np.ndarray:
        """Predict binary labels."""
        proba = self.predict_proba(X)[:, 1]
        return (proba >= threshold).astype(int)
    
    def evaluate(self, X: pd.DataFrame, y: pd.Series) -> Dict:
        """Evaluate ensemble performance."""
        y_pred_proba = self.predict_proba(X)[:, 1]
        y_pred = (y_pred_proba >= 0.5).astype(int)
        
        metrics = {
            'ROC-AUC': roc_auc_score(y, y_pred_proba),
            'Precision': precision_score(y, y_pred),
            'Recall': recall_score(y, y_pred),
            'F1': f1_score(y, y_pred),
            'Confusion Matrix': confusion_matrix(y, y_pred)
        }
        
        logger.info("=" * 50)
        logger.info("ENSEMBLE PERFORMANCE")
        logger.info("=" * 50)
        logger.info(f"ROC-AUC: {metrics['ROC-AUC']:.4f}")
        logger.info(f"Precision: {metrics['Precision']:.4f}")
        logger.info(f"Recall: {metrics['Recall']:.4f}")
        logger.info(f"F1-Score: {metrics['F1']:.4f}")
        logger.info(classification_report(y, y_pred))
        
        return metrics
    
    def get_feature_importance(self) -> pd.DataFrame:
        """Extract feature importance from tree models."""
        importances = {}
        
        if self.xgboost is not None:
            xgb_imp = self.xgboost.feature_importances_
            importances['XGBoost'] = xgb_imp
        
        if self.catboost is not None:
            cat_imp = self.catboost.feature_importances_
            importances['CatBoost'] = cat_imp
        
        if not importances:
            logger.warning("No tree models available for feature importance")
            return pd.DataFrame()
        
        imp_df = pd.DataFrame(importances, index=self.feature_names)
        imp_df['Mean'] = imp_df.mean(axis=1)
        imp_df = imp_df.sort_values('Mean', ascending=False)
        
        return imp_df
    
    def save(self, filepath: str) -> None:
        """Save trained ensemble."""
        joblib.dump(self, filepath)
        logger.info(f"Ensemble saved to {filepath}")
    
    @staticmethod
    def load(filepath: str) -> 'HybridEnsemble':
        """Load trained ensemble."""
        ensemble = joblib.load(filepath)
        logger.info(f"Ensemble loaded from {filepath}")
        return ensemble


In [ ]:
# Train the models
def stage_model_training(config: dict, features: pd.DataFrame,
                        target: pd.Series):
    """Stage 3: Train ensemble model."""
    logger.info("\n" + "="*60)
    logger.info("STAGE 3: MODEL TRAINING")
    logger.info("="*60)
    
    from src.models.ensemble_model import HybridEnsemble
    
    # Align indices
    valid_idx = features.index.intersection(target.index)
    X = features.loc[valid_idx].dropna()
    y = target.loc[X.index]
    
    logger.info(f"Training data: {X.shape}")
    
    # Train ensemble
    ensemble = HybridEnsemble(config)
    ensemble.fit(X, y, validation_split=0.2)
    
    # Evaluate
    metrics = ensemble.evaluate(X, y)
    
    # Feature importance
    imp_df = ensemble.get_feature_importance()
    if not imp_df.empty:
        logger.info("\nTop 10 Features:")
        logger.info(imp_df.head(10))
        imp_df.to_csv('data/processed/feature_importance.csv')
    
    # Save model
    os.makedirs('data/results', exist_ok=True)
    ensemble.save('data/results/ensemble_model.pkl')
    logger.info("✓ Model saved")
    
    return ensemble

In [8]:
# Read strategy results from csv file and log the results
def read_strategy_results(file_path: str) -> pd.DataFrame:
    """Read strategy results from CSV file."""
    df = pd.read_csv(file_path)
    logger.info(f"Strategy results loaded: {df.shape}")
    return df
result = read_strategy_results('data/results/strategy_results.csv')
result.tail()

,Date,Position,Signal,Prediction,Strategy_Return,SPY_Return,SPY_Price,Strategy_Cumulative,SPY_Cumulative
5218,2026-01-09,0.787517,1,0.106242,0.005191,0.006592,694.070007,193647.973473,589.683218
5219,2026-01-12,0.792028,1,0.103986,0.001243,0.001569,695.159973,193888.643705,590.608529
5220,2026-01-13,0.772290,1,0.113855,-0.001546,-0.002001,693.770020,193588.946249,589.426441
5221,2026-01-14,0.779427,1,0.110287,-0.003841,-0.004927,690.359985,192845.467290,586.522135
5222,2026-01-15,0.755418,1,0.122291,0.002054,0.002720,692.239990,193241.644232,588.117196
